In [11]:
from pathlib import Path
import sys
import pandas as pd
import yaml

root_path = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").exists()), None)
if root_path is None:
    raise FileNotFoundError("Could not locate project root containing 'utils' directory")
sys.path.append(str(root_path))

from utils.helpers import save_csv_file, load_data
from utils.helper_CBM import (
    get_cbm_valid_questions,
    cast_columns_to_string,
    filter_cbm_questions,
    drop_columns_from_config,
    add_auto_score,
    split_response_cbm,
    map_confidence_level,
    validate_and_remove_invalid_confidence
)


In [12]:
df = load_data("../data/cleaned_data/IN1020_2025konte_cleaned_data.csv")
df.head()

Data loaded successfully from ../data/cleaned_data/IN1020_2025konte_cleaned_data.csv


,exam_start_time,exam_end_time,extra_time_mins,incident_time_mins,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,candidate_response_code,candidate_response_text,response_CBM,total_score
0,2026-01-22 14:01:25,2026-01-22 15:33:03,0,0,38,2.0,1.1,IN1020 K-H25 bit-representasjon,225,2.0,simpleChoice_1368096164456,-85,2,68.25
1,2026-01-22 14:01:25,2026-01-22 15:33:03,0,0,38,2.0,1.1,IN1020 K-H25 bit-representasjon,225,2.0,simpleChoice_1368096164453,171,1,68.25
2,2026-01-22 14:01:25,2026-01-22 15:33:03,0,0,38,3.5,1.2,IN1020 K-H25 Tallsystemer representasjon,97,3.0,simpleChoice_1368096164456,cafebabe,2,68.25
3,2026-01-22 14:01:25,2026-01-22 15:33:03,0,0,38,3.5,1.2,IN1020 K-H25 Tallsystemer representasjon,97,3.0,simpleChoice_IA17639200973561dd93152-292d-4b6b...,123hh321,4,68.25
4,2026-01-22 14:01:25,2026-01-22 15:33:03,0,0,38,3.5,1.2,IN1020 K-H25 Tallsystemer representasjon,97,3.0,simpleChoice_1368096164453,0f0cdb0d,1,68.25


In [13]:
config_path = root_path / "configs" / "CBM_config.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
print(f"Config loaded from {config_path}")


Config loaded from /Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/configs/CBM_config.yaml


In [14]:
df = cast_columns_to_string(df, config.get('string_columns', []))

valid_questions = get_cbm_valid_questions(prefix="2.", start=1, end=15)
df_CBM = filter_cbm_questions(df, valid_questions)
df_CBM = drop_columns_from_config(df_CBM, config.get('columns_to_remove', []))

df_CBM.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4862 entries, 21 to 10123
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   student_id               4862 non-null   int64   
 1   max_question_score       4862 non-null   float64 
 2   question_number          4862 non-null   category
 3   question_title           4862 non-null   object  
 4   question_duration_sec    4862 non-null   int64   
 5   auto_score_per_question  4862 non-null   float64 
 6   response_CBM             4862 non-null   object  
 7   total_score              4862 non-null   float64 
dtypes: category(1), float64(3), int64(2), object(2)
memory usage: 309.3+ KB


In [15]:
df_CBM = add_auto_score(df_CBM)
df_CBM.head(10)


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,response_CBM,total_score,auto_score
21,38,1.0,2.1,H25K - CBA - PC,32,1.0,1 5,68.25,20.0
22,38,1.0,2.1,H25K - CBA - PC,32,1.0,4 1,68.25,20.0
23,38,1.0,2.1,H25K - CBA - PC,32,1.0,3 1,68.25,20.0
24,38,1.0,2.1,H25K - CBA - PC,32,1.0,5 1,68.25,20.0
25,38,1.0,2.1,H25K - CBA - PC,32,1.0,2 1,68.25,20.0
26,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,3 1,68.25,20.0
27,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,4 5,68.25,20.0
28,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,5 1,68.25,20.0
29,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,1 1,68.25,20.0
30,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,2 1,68.25,20.0


In [16]:
df_CBM = split_response_cbm(df_CBM, config)
df_CBM.head(10)


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
21,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,1,5
22,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,4,1
23,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,3,1
24,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,5,1
25,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,2,1
26,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,3,1
27,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,4,5
28,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,5,1
29,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,1,1
30,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,2,1


In [17]:
df_CBM = map_confidence_level(df_CBM)
df_CBM.head(10)


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
21,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,1,100.0
22,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,4,0.0
23,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,3,0.0
24,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,5,0.0
25,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,2,0.0
26,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,3,0.0
27,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,4,100.0
28,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,5,0.0
29,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,1,0.0
30,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,2,0.0


In [18]:
df_CBM = validate_and_remove_invalid_confidence(df_CBM)
df_CBM.head(10)

Found 11 student(s) with invalid confidence sums across 15 question(s):
  Question 2.1: 3 student(s) with incorrect sum
  Question 2.2: 2 student(s) with incorrect sum
  Question 2.3: 5 student(s) with incorrect sum
  Question 2.4: 0 student(s) with incorrect sum
  Question 2.5: 3 student(s) with incorrect sum
  Question 2.6: 2 student(s) with incorrect sum
  Question 2.7: 2 student(s) with incorrect sum
  Question 2.8: 4 student(s) with incorrect sum
  Question 2.9: 5 student(s) with incorrect sum
  Question 2.10: 3 student(s) with incorrect sum
  Question 2.11: 4 student(s) with incorrect sum
  Question 2.12: 3 student(s) with incorrect sum
  Question 2.13: 3 student(s) with incorrect sum
  Question 2.14: 3 student(s) with incorrect sum
  Question 2.15: 4 student(s) with incorrect sum


/Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/utils/helper_CBM.py:183: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([student_col, question_col])[confidence_col]
/Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/utils/helper_CBM.py:197: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  invalid.groupby(question_col)[student_col]


,student_id,max_question_score,question_number,question_title,question_duration_sec,auto_score_per_question,total_score,auto_score,answer_alternative,confidence_level
0,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,1,100.0
1,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,4,0.0
2,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,3,0.0
3,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,5,0.0
4,38,1.0,2.1,H25K - CBA - PC,32,1.0,68.25,20.0,2,0.0
5,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,3,0.0
6,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,4,100.0
7,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,5,0.0
8,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,1,0.0
9,38,1.0,2.2,H25K - CBA - Datapath,141,1.0,68.25,20.0,2,0.0


In [19]:
output_folder = root_path / "analysis" / "CBM" / "2025" / "Konte" / "data" / "CBM_format"
save_csv_file(df_CBM, "IN1020_2025konte_CBM", output_folder)

✅ DataFrame saved as: /Users/yulis/Desktop/Masteroppgave_Eksamensanalyse/analysis/CBM/2025/Konte/data/CBM_format/IN1020_2025konte_CBM.csv
